# COVID-19 Logistic Regression Training Script

This notebook cleans and preprocesses the COVID-19 testing dataset, then trains and tunes a **Logistic Regression** classifier, evaluates it, and saves a single, fully self-contained deployment file: the fitted preprocessor and the tuned model bundled together as a scikit-learn `Pipeline`, alongside this model's test metrics. It is one of three independent training scripts — run `train_random_forest.ipynb` and `train_linear_svm.ipynb` separately to produce the other two model files. Each file is fully self-contained, so no separate preprocessor or metrics file is needed; `app.py` loads all three files directly.

**Dataset file:** `corona_tested_individuals_ver_006.english.csv.zip`  
**Evaluation priority:** F1-score, because positive cases are the minority class

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sklearn

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Seaborn version:", sns.__version__)
print("Scikit-learn version:", sklearn.__version__)

print("\nEnvironment is ready!")

In [ ]:
from pathlib import Path

dataset_filename = "corona_tested_individuals_ver_006.english.csv.zip"
dataset_url = (
    "https://raw.githubusercontent.com/"
    "nshomron/covidpred/master/data/"
    + dataset_filename
)

local_dataset = Path.cwd() / dataset_filename
dataset_source = (
    local_dataset
    if local_dataset.exists()
    else dataset_url
)

try:
    df = pd.read_csv(
        dataset_source,
        compression="zip",
        low_memory=False
    )
except Exception as error:
    raise RuntimeError(
        "Dataset loading failed. Check the internet connection "
        "or place the ZIP dataset in the same folder as this notebook."
    ) from error

required_columns = {
    "test_date",
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache",
    "corona_result",
    "age_60_and_above",
    "gender",
    "test_indication"
}

missing_columns = required_columns - set(df.columns)

if df.empty:
    raise ValueError("The dataset is empty.")

if missing_columns:
    raise ValueError(
        "Dataset columns are missing: "
        + ", ".join(sorted(missing_columns))
    )

print("Dataset loaded successfully!")
print("Dataset source:", dataset_source)
print("Dataset shape:", df.shape)

In [ ]:
display(df.head())

In [ ]:
print("Column names:")

for column in df.columns:
    print("-", column)

In [ ]:
df.info()

In [ ]:
missing_values = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

missing_table = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage (%)": missing_percentage
})

display(
    missing_table.sort_values(
        by="Missing Values",
        ascending=False
    )
)

In [ ]:
print("Corona result distribution:")
print(df["corona_result"].value_counts(dropna=False))

In [ ]:
categorical_columns = [
    "age_60_and_above",
    "gender",
    "test_indication"
]

for column in categorical_columns:
    print(f"\nColumn: {column}")
    print(df[column].value_counts(dropna=False))

In [ ]:
clean_df = df.copy()

print("Original dataset shape:", clean_df.shape)

In [ ]:
text_columns = [
    "corona_result",
    "age_60_and_above",
    "gender",
    "test_indication"
]

for column in text_columns:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

print("Text values standardized successfully!")

In [ ]:
clean_df = clean_df[
    clean_df["corona_result"].isin([
        "positive",
        "negative"
    ])
].copy()

print("Records after removing 'other':", len(clean_df))

print("\nRemaining target values:")
print(clean_df["corona_result"].value_counts())

In [ ]:
clean_df["target"] = clean_df["corona_result"].map({
    "negative": 0,
    "positive": 1
})

print(clean_df[[
    "corona_result",
    "target"
]].head(10))

In [ ]:
categorical_columns = [
    "age_60_and_above",
    "gender",
    "test_indication"
]

clean_df[categorical_columns] = (
    clean_df[categorical_columns]
    .fillna("unknown")
)

print("Categorical missing values handled!")

In [ ]:
symptom_columns = [
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache"
]

for column in symptom_columns:
    # Make sure the values are numeric
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

    # Find the most frequently occurring value
    most_common_value = clean_df[column].mode()[0]

    # Fill missing values
    clean_df[column] = (
        clean_df[column]
        .fillna(most_common_value)
        .astype(int)
    )

    print(
        column,
        "filled with:",
        most_common_value
    )

In [ ]:
clean_df = clean_df.drop(
    columns=[
        "test_date",
        "corona_result"
    ]
)

print("Unused columns removed!")

In [ ]:
print("Cleaned dataset shape:", clean_df.shape)

print("\nMissing values after preprocessing:")
print(clean_df.isna().sum())

print("\nTarget distribution:")
print(clean_df["target"].value_counts())

display(clean_df.head())

In [ ]:
# X contains the input features
X = clean_df.drop(columns=["target"])

# y contains the correct answer
y = clean_df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nInput features:")
print(X.columns.tolist())

In [ ]:
class_distribution = pd.DataFrame({
    "Count": y.value_counts(),
    "Percentage (%)": (
        y.value_counts(normalize=True) * 100
    ).round(2)
})

class_distribution.index = [
    "Negative (0)",
    "Positive (1)"
]

display(class_distribution)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

In [ ]:
print("Training target distribution:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
symptom_features = [
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache"
]

categorical_features = [
    "age_60_and_above",
    "gender",
    "test_indication"
]

print("Symptom features:", symptom_features)
print("Categorical features:", categorical_features)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "symptoms",
            "passthrough",
            symptom_features
        ),
        (
            "categories",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessor created successfully!")

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)

X_test_encoded = preprocessor.transform(X_test)

print(
    "Encoded training shape:",
    X_train_encoded.shape
)

print(
    "Encoded testing shape:",
    X_test_encoded.shape
)

In [ ]:
encoded_feature_names = (
    preprocessor.get_feature_names_out()
)

print("Total encoded features:", len(encoded_feature_names))

for feature in encoded_feature_names:
    print("-", feature)

### 5.1 Encoded Data Validation

The following checks ensure that preprocessing produced valid numeric data before model training begins. These validations help prevent missing values, inconsistent sample counts, and incomplete target classes from entering the models.

In [ ]:
if X_train_encoded.shape[0] != len(y_train):
    raise ValueError("Training feature and target counts do not match.")

if X_test_encoded.shape[0] != len(y_test):
    raise ValueError("Testing feature and target counts do not match.")

if np.isnan(X_train_encoded).any() or np.isnan(X_test_encoded).any():
    raise ValueError("Encoded data contains missing numeric values.")

if set(y_train.unique()) != {0, 1}:
    raise ValueError("The training target must contain classes 0 and 1.")

print("Encoded data validation passed!")
print("Training samples:", X_train_encoded.shape[0])
print("Testing samples:", X_test_encoded.shape[0])
print("Encoded features:", X_train_encoded.shape[1])

## Part 6: Model Development and Hyperparameter Tuning

Each algorithm is developed in two stages:

1. A **baseline model** provides an initial result.
2. A **tuned model** is selected using cross-validation on the training set.

The untouched test set is used only for the final baseline and tuned evaluations. F1-score is the tuning metric because the dataset is highly imbalanced.

In [ ]:
import time
import joblib
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    RandomizedSearchCV,
    StratifiedKFold
)
from sklearn.svm import LinearSVC

stratified_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print("Model development tools imported successfully!")

In [ ]:
def evaluate_classifier(
    model_name,
    model,
    X_evaluation,
    y_evaluation,
    training_time,
    colour_map="Blues"
):
    """Evaluate one fitted classifier using consistent metrics."""
    predictions = model.predict(X_evaluation)

    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_evaluation, predictions),
        "Precision": precision_score(
            y_evaluation,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_evaluation,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_evaluation,
            predictions,
            zero_division=0
        ),
        "Training Time": training_time
    }

    print(model_name)
    print("-" * len(model_name))
    print(f"Accuracy : {result['Accuracy']:.4f}")
    print(f"Precision: {result['Precision']:.4f}")
    print(f"Recall   : {result['Recall']:.4f}")
    print(f"F1-score : {result['F1-score']:.4f}")
    print(f"Fit time : {training_time:.2f} seconds")
    print()
    print(
        classification_report(
            y_evaluation,
            predictions,
            target_names=["Negative", "Positive"],
            digits=4,
            zero_division=0
        )
    )

    ConfusionMatrixDisplay.from_predictions(
        y_evaluation,
        predictions,
        display_labels=["Negative", "Positive"],
        cmap=colour_map,
        values_format="d"
    )
    plt.title(f"{model_name} Confusion Matrix")
    plt.show()

    return result, predictions


def display_search_summary(search_object, top_n=5):
    """Display the highest-ranked cross-validation combinations."""
    results = pd.DataFrame(search_object.cv_results_)
    summary = (
        results[[
            "params",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]]
        .sort_values("rank_test_score")
        .head(top_n)
        .reset_index(drop=True)
    )
    display(summary)
    return summary

### 6.1 Logistic Regression

Logistic Regression is first trained with a reasonable baseline iteration limit. GridSearchCV then tunes regularisation strength (`C`), solver, and `max_iter`.

In [ ]:
baseline_logistic_model = LogisticRegression(
    max_iter=100,
    class_weight="balanced",
    random_state=42
)

start_time = time.time()
baseline_logistic_model.fit(X_train_encoded, y_train)
baseline_logistic_training_time = time.time() - start_time

baseline_logistic_result, baseline_logistic_predictions = (
    evaluate_classifier(
        "Baseline Logistic Regression",
        baseline_logistic_model,
        X_test_encoded,
        y_test,
        baseline_logistic_training_time,
        "Blues"
    )
)

In [ ]:
logistic_parameter_grid = {
    "solver": ["liblinear", "lbfgs"],
    "C": [0.01, 0.1, 1.0, 10.0],
    "max_iter": [100, 200]
}

logistic_grid_search = GridSearchCV(
    estimator=LogisticRegression(
        class_weight="balanced",
        random_state=42
    ),
    param_grid=logistic_parameter_grid,
    scoring="f1",
    cv=stratified_cv,
    n_jobs=-1,
    return_train_score=True,
    refit=True
)

logistic_combination_count = len(
    list(ParameterGrid(logistic_parameter_grid))
)

print("Parameter combinations:", logistic_combination_count)
print("Total cross-validation fits:", logistic_combination_count * 3)

In [ ]:
logistic_tuning_start = time.time()
logistic_grid_search.fit(X_train_encoded, y_train)
logistic_tuning_time = time.time() - logistic_tuning_start

tuned_logistic_model = logistic_grid_search.best_estimator_

print("Logistic Regression tuning completed!")
print("Best parameters:", logistic_grid_search.best_params_)
print(f"Best cross-validation F1: {logistic_grid_search.best_score_:.4f}")
print(f"Total tuning time: {logistic_tuning_time:.2f} seconds")
print(
    "Iterations used by selected model:",
    int(np.max(tuned_logistic_model.n_iter_))
)

logistic_cv_summary = display_search_summary(
    logistic_grid_search
)

In [ ]:
tuned_logistic_result, tuned_logistic_predictions = (
    evaluate_classifier(
        "Logistic Regression",
        tuned_logistic_model,
        X_test_encoded,
        y_test,
        logistic_grid_search.refit_time_,
        "Greens"
    )
)

logistic_before_after = pd.DataFrame([
    baseline_logistic_result,
    tuned_logistic_result
])

display(
    logistic_before_after.round({
        "Accuracy": 4,
        "Precision": 4,
        "Recall": 4,
        "F1-score": 4,
        "Training Time": 2
    })
)

print(
    "F1-score change:",
    f"{tuned_logistic_result['F1-score'] - baseline_logistic_result['F1-score']:+.4f}"
)

## Save Logistic Regression for Deployment

Bundles the shared preprocessor and the tuned Logistic Regression model together as a scikit-learn `Pipeline`, then saves that pipeline together with this model's test metrics in a single, self-contained file. No separate `preprocessor.pkl` or `metrics.pkl` is written.

In [ ]:
from sklearn.pipeline import Pipeline

model_dir = Path.cwd()

logistic_regression_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", tuned_logistic_model)
])

model_path = model_dir / "model_logistic_regression.pkl"
joblib.dump(
    {
        "pipeline": logistic_regression_pipeline,
        "metrics": {
            "Accuracy": tuned_logistic_result["Accuracy"],
            "Precision": tuned_logistic_result["Precision"],
            "Recall": tuned_logistic_result["Recall"],
            "F1-score": tuned_logistic_result["F1-score"],
            "Training Time": tuned_logistic_result["Training Time"]
        }
    },
    model_path
)

print("Logistic Regression saved successfully!")
print(
    "Saved file:", model_path.name,
    "(pipeline + metrics, fully self-contained)"
)

In [ ]:
verification_entry = joblib.load(model_dir / "model_logistic_regression.pkl")

required_entry_keys = {"pipeline", "metrics"}
missing_entry_keys = required_entry_keys - set(verification_entry)
if missing_entry_keys:
    raise KeyError(
        "Logistic Regression file is missing: "
        + ", ".join(sorted(missing_entry_keys))
    )

verification_pipeline = verification_entry["pipeline"]
verification_metrics = verification_entry["metrics"]

if not hasattr(verification_pipeline, "predict"):
    raise TypeError("The saved Logistic Regression pipeline is invalid.")

required_metrics = {
    "Accuracy", "Precision", "Recall", "F1-score", "Training Time"
}
missing_metrics = required_metrics - set(verification_metrics)
if missing_metrics:
    raise KeyError(
        "Logistic Regression metrics were not saved correctly: "
        + ", ".join(sorted(missing_metrics))
    )

# Use a RAW (unencoded) sample here — the pipeline does its own
# preprocessing internally
sample_raw_input = X_test.iloc[[0]]
sample_prediction = verification_pipeline.predict(sample_raw_input)

print("Logistic Regression verification prediction:", int(sample_prediction[0]))
print("Saved metrics for this model:", verification_metrics)
print("File is self-contained and compatible with app.py!")